# Homework 2 - Distributed Data Managment

You can copy the whole notebook to your Google account and work on it on Colab via:

File -> Save a copy in Drive -> Open the copied notebook

Install findspark, pyspark in case it is not installed - if running on Colab.

In [ ]:
# * When using the Docker local workspace do not run this step *
IM_RUNNNING_ON_COLAB = True

if IM_RUNNNING_ON_COLAB:

  !pip install --force-reinstall pyspark==3.4
  !pip install findspark


Upload the data from Moodle, it's a zip file so simply unzip it



In [ ]:
!unzip /content/random_data.parquet.zip

#### SparkSession is created outside your function

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
import pyspark
from time import time

def init_spark(app_name: str):
  spark = SparkSession.builder.appName(app_name).getOrCreate()
  sc = spark.sparkContext
  return spark, sc

spark, sc = init_spark('hw2_kmeans_24')

#### Load samples points

In [ ]:
import os

data_df = spark.read.parquet("/content/random_data.parquet")
data_df.show(5)

# You can load the small sample for quick testing and reproducing results:

sample_df = spark.read.option("header",True) \
                    .option('inferSchema', True)\
                    .csv('sample_data_84.csv')
sample_df.show(5)

#### Create initials centroids

In [ ]:
init_centroids = \
  spark.createDataFrame([[5.05, 6.06, 7.07],
                         [3.79, 4.65, 7.31],
                         [3.14, 1.609, 4.20],
                         [8.04, 8.47, 9.09]])
init_centroids.show()

#### Place your kmeans_fit function here
#### Don't forget to also add it in a seperate .py file named HW2_WET_[ID1]_[ID1]
Example:
HW2_WET_000000000_000000000.py

In [ ]:
# %%file HW2_WET_000000000_000000000.py
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import DoubleType, ArrayType, StructType, StructField
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import VectorAssembler
from pyspark import SparkContext


def euclidean_distance(v1, v2):
    return float(sum((a - b) ** 2 for a, b in zip(v1, v2)) ** 0.5)

def kmeans_fit(data: DataFrame,
               init: DataFrame,
               k: int = 4,
               max_iter: int = 10):

    centroids = [Vectors.dense(row['_1'], row['_2'], row['_3']) for row in init.collect()]
    centroids_broadcast = sc.broadcast(centroids)
    assembler = VectorAssembler(inputCols=['_1', '_2', '_3'], outputCol='features')
    data = assembler.transform(data)

    for iteration in range(max_iter):
      def assign_cluster(features):
            centroids = centroids_broadcast.value
            distances = [euclidean_distance(features.toArray(), centroid.toArray()) for centroid in centroids]
            return float(distances.index(min(distances)))

      assign_cluster_udf = udf(assign_cluster, DoubleType())
      data_with_clusters = data.withColumn('cluster', assign_cluster_udf(col('features')))
      new_centroids = data_with_clusters.groupBy('cluster').agg({'_1': 'avg', '_2': 'avg', '_3': 'avg'})
      new_centroids_rdd = new_centroids.rdd.map(lambda row: Vectors.dense(row['avg(_1)'], row['avg(_2)'], row['avg(_3)']))
      new_centroids_vector = new_centroids_rdd.collect()
      num_converged = 0

      for new_centroid in new_centroids_vector:
          min_distance = min(euclidean_distance(new_centroid, old_centroid) for old_centroid in centroids_broadcast.value)
          if min_distance < 0.001:
            num_converged += 1

      if num_converged == k:
        centroids = new_centroids_vector
        centroids_broadcast = sc.broadcast(centroids)
        break

      centroids = new_centroids_vector
      centroids_broadcast = sc.broadcast(centroids)

    schema = StructType([StructField("centroids", ArrayType(DoubleType()), False)])
    final_centroids_df = spark.createDataFrame(
        [(list(map(float, centroid.toArray())),) for centroid in centroids],
        schema=schema
    )

    return final_centroids_df
    pass

#### Test your function output and run time

In [ ]:
start_time = time()
out = kmeans_fit(data_df, init_centroids)
end_time = time()

print('Final results:')
out.show(truncate=False)
print(f'Total runtime: {end_time-start_time:.3f} seconds')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Expected results
For the given intialization centroids and the sample_df_84 you got on Moodle, the expected results is:



In [ ]:
#  +----------------------------------------------------------+
#  |centroids                                                 |
#  +----------------------------------------------------------+
#  |[4.496670602125162,4.495811688311686,4.50180401416766]    |
#  |[1.4891561106155216,1.5075798394290807,1.4981257805530763]|
#  |[8.501673279603231,8.490239925604461,8.505297582145058]   |
#  |[6.496611142694714,6.508118249005111,6.510762933484945]   |
#  +----------------------------------------------------------+


Notice that the algorithm is deterministic, and we **expect your results to be the same for at least 3 decimal numbers** after the point.

We expect your output dataframe to contain a single column named 'centroids', and each row to contain a single dense Vector representing a centroid.

The ordering of the centroids in the Dataframe and the title may be different.

*Helpful hint:* You may use VectorAssembler. https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.VectorAssembler.html


# **Don't forget to:**
* ## Run your function on the LARGE data file and show the results in the notebook's PDF and HTML
* ## Add your function & imports in a seperate .py file named HW2_WET_[ID1]_[ID1].py
Example:
HW2_WET_000000000_000000000.py